# 05 — Causal Analysis

**Project:** Causal ML for Climate Policy Effectiveness  
**Purpose:** Estimate average and heterogeneous treatment effects using DML & Causal Forest  
**Input:** `data/processed/analysis_panel.csv`  
**Output:** Causal estimates, CATEs, model comparison tables

---

## Methods
1. **Baseline:** OLS with country and year fixed effects  
2. **Double/Debiased ML (DML):** Flexible first-stage models (XGBoost) with cross-fitting  
3. **Causal Forest:** Heterogeneous treatment effects via Generalized Random Forests  
4. **Multi-treatment analysis:** Separate effects for carbon pricing, renewable mandates, efficiency standards

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.causal_models import run_ols_baseline, run_dml, run_causal_forest, compare_methods
from src.visualization import set_publication_style, save_figure, COLORS

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
set_publication_style()

In [2]:
# Load analysis panel
panel = pd.read_csv(PROCESSED_DIR / "analysis_panel.csv")
print(f"Panel: {panel.shape[0]:,} rows × {panel.shape[1]} columns")
print(f"Countries: {panel['iso3'].nunique()}, Years: {panel['year'].min()}-{panel['year'].max()}")

Panel: 7,630 rows × 77 columns
Countries: 218, Years: 1990-2024


In [3]:
# Define analysis variables
OUTCOME = "log_co2"
TREATMENTS = [c for c in panel.columns if c.startswith("treat_") and panel[c].std() > 0]

# Define confounders — use available columns
potential_covariates = [
    "population", "gdp_per_capita", "urban_pct", "co2_lag1"
]
COVARIATES = [c for c in potential_covariates if c in panel.columns]

# Ensure consistent sample for all methods
required_cols = [OUTCOME] + TREATMENTS + COVARIATES + ["regulatory_quality", "renewables_share_energy"]
required_cols = [c for c in required_cols if c in panel.columns]
panel = panel.dropna(subset=required_cols).copy()

print(f"Outcome: {OUTCOME}")
print(f"Treatments: {TREATMENTS}")
print(f"Covariates ({len(COVARIATES)}): {COVARIATES}")
print(f"Sample size after dropping missing values across all required columns: {len(panel)}")

Outcome: log_co2
Treatments: ['treat_carbon_pricing', 'treat_renewable_mandate', 'treat_efficiency_standard', 'treat_t1', 'treat_t3', 'treat_t5']
Covariates (4): ['population', 'gdp_per_capita', 'urban_pct', 'co2_lag1']
Sample size after dropping missing values across all required columns: 2028


---
## 5.1 — Baseline: OLS Fixed Effects

In [4]:
print("=" * 60)
print("5.1 — Baseline OLS Estimates")
print("=" * 60)

ols_results = []
for treat in TREATMENTS:
    try:
        est = run_ols_baseline(panel, OUTCOME, treat, COVARIATES, fe_cols=["iso3", "year"])
        ols_results.append(est)
        sig = "***" if est.is_significant else ""
        print(f"  {treat:30s}: {est.estimate:+.4f} [{est.ci_lower:.4f}, {est.ci_upper:.4f}] {sig}")
    except Exception as e:
        print(f"  {treat:30s}: FAILED — {e}")

if ols_results:
    print(f"\n{compare_methods(ols_results).to_string(index=False)}")

5.1 — Baseline OLS Estimates


  treat_carbon_pricing          : +0.1936 [0.1551, 0.2321] ***
  treat_renewable_mandate       : +0.1055 [0.0684, 0.1427] ***
  treat_efficiency_standard     : +0.2165 [0.1672, 0.2657] ***
  treat_t1                      : +0.1627 [0.1254, 0.1999] ***


  treat_t3                      : +0.1181 [0.0822, 0.1541] ***
  treat_t5                      : +0.0556 [0.0208, 0.0904] ***

Method                 Treatment Estimate Std. Error           95% CI p-value    N Significant
   OLS      treat_carbon_pricing   0.1936     0.0196 [0.1551, 0.2321]  0.0000 2028         ***
   OLS   treat_renewable_mandate   0.1055     0.0190 [0.0684, 0.1427]  0.0000 2028         ***
   OLS treat_efficiency_standard   0.2165     0.0251 [0.1672, 0.2657]  0.0000 2028         ***
   OLS                  treat_t1   0.1627     0.0190 [0.1254, 0.1999]  0.0000 2028         ***
   OLS                  treat_t3   0.1181     0.0184 [0.0822, 0.1541]  0.0000 2028         ***
   OLS                  treat_t5   0.0556     0.0178 [0.0208, 0.0904]  0.0017 2028         ***


---
## 5.2 — Double/Debiased ML (DML)

In [5]:
print("=" * 60)
print("5.2 — Double/Debiased ML Estimates")
print("=" * 60)

dml_results = []
for treat in TREATMENTS:
    print(f"\n  Estimating DML for: {treat}")
    try:
        est = run_dml(panel, OUTCOME, treat, COVARIATES,
                      n_folds=5, first_stage_model="xgboost")
        dml_results.append(est)
        sig = "***" if est.is_significant else ""
        print(f"    ATE: {est.estimate:+.4f} [{est.ci_lower:.4f}, {est.ci_upper:.4f}] {sig}")
        print(f"    N={est.n_obs:,}, Treated={est.n_treated:,}")
    except Exception as e:
        print(f"    FAILED: {e}")

if dml_results:
    print(f"\n{compare_methods(dml_results).to_string(index=False)}")

5.2 — Double/Debiased ML Estimates

  Estimating DML for: treat_carbon_pricing


    ATE: +0.0899 [0.0598, 0.1200] ***
    N=2,028, Treated=0

  Estimating DML for: treat_renewable_mandate


    ATE: +0.0203 [-0.0077, 0.0482] 
    N=2,028, Treated=0

  Estimating DML for: treat_efficiency_standard


    ATE: +0.0537 [0.0155, 0.0919] ***
    N=2,028, Treated=0

  Estimating DML for: treat_t1


    ATE: +0.0776 [0.0494, 0.1057] ***
    N=2,028, Treated=0

  Estimating DML for: treat_t3


    ATE: +0.0677 [0.0404, 0.0951] ***
    N=2,028, Treated=0

  Estimating DML for: treat_t5


    ATE: +0.0175 [-0.0098, 0.0449] 
    N=2,028, Treated=0

       Method                 Treatment Estimate Std. Error            95% CI p-value    N Significant
DML (xgboost)      treat_carbon_pricing   0.0899     0.0154  [0.0598, 0.1200]  0.0000 2028         ***
DML (xgboost)   treat_renewable_mandate   0.0203     0.0143 [-0.0077, 0.0482]  0.1549 2028            
DML (xgboost) treat_efficiency_standard   0.0537     0.0195  [0.0155, 0.0919]  0.0059 2028         ***
DML (xgboost)                  treat_t1   0.0776     0.0144  [0.0494, 0.1057]  0.0000 2028         ***
DML (xgboost)                  treat_t3   0.0677     0.0139  [0.0404, 0.0951]  0.0000 2028         ***
DML (xgboost)                  treat_t5   0.0175     0.0140 [-0.0098, 0.0449]  0.2097 2028            


In [6]:
# DML with alternative first-stage (Random Forest)
print("\n--- DML with Random Forest first stage ---")

dml_rf_results = []
primary_treat = TREATMENTS[0] if TREATMENTS else None

if primary_treat:
    for model_type in ["xgboost", "random_forest", "gradient_boosting"]:
        try:
            est = run_dml(panel, OUTCOME, primary_treat, COVARIATES,
                          n_folds=5, first_stage_model=model_type)
            dml_rf_results.append(est)
            print(f"  {model_type:25s}: {est.estimate:+.4f} [{est.ci_lower:.4f}, {est.ci_upper:.4f}]")
        except Exception as e:
            print(f"  {model_type:25s}: FAILED — {e}")


--- DML with Random Forest first stage ---


  xgboost                  : +0.0899 [0.0598, 0.1200]


  random_forest            : +0.0948 [0.0633, 0.1263]


  gradient_boosting        : +0.0868 [0.0544, 0.1192]


---
## 5.3 — Causal Forest: Heterogeneous Effects

In [7]:
print("=" * 60)
print("5.3 — Causal Forest (Heterogeneous Treatment Effects)")
print("=" * 60)

if primary_treat and primary_treat in panel.columns:
    # Effect modifiers — variables we think might moderate the treatment effect
    effect_modifiers = [c for c in ["gdp_per_capita", "regulatory_quality",
                                     "renewables_share_energy", "urban_pct"]
                        if c in panel.columns]
    
    if not effect_modifiers:
        effect_modifiers = COVARIATES[:4]  # fallback
    
    print(f"  Treatment: {primary_treat}")
    print(f"  Effect modifiers: {effect_modifiers}")
    
    try:
        cf_ate, cate_df = run_causal_forest(
            panel, OUTCOME, primary_treat, COVARIATES,
            effect_modifier_cols=effect_modifiers,
            n_estimators=500,
        )
        print(f"\n  ATE from Causal Forest: {cf_ate.estimate:+.4f} [{cf_ate.ci_lower:.4f}, {cf_ate.ci_upper:.4f}]")
        print(f"  CATE range: [{cate_df['cate'].min():.4f}, {cate_df['cate'].max():.4f}]")
        print(f"  CATE std: {cate_df['cate'].std():.4f}")
        
        # Save CATEs
        cate_df.to_csv(PROCESSED_DIR / "cate_estimates.csv", index=False)
        print(f"   CATEs saved to {PROCESSED_DIR / 'cate_estimates.csv'}")
    except Exception as e:
        print(f"   Causal Forest failed: {e}")
        cf_ate = None
        cate_df = None

5.3 — Causal Forest (Heterogeneous Treatment Effects)
  Treatment: treat_carbon_pricing
  Effect modifiers: ['gdp_per_capita', 'regulatory_quality', 'renewables_share_energy', 'urban_pct']



  ATE from Causal Forest: +0.0988 [-0.0345, 0.2321]
  CATE range: [-0.2695, 0.4556]
  CATE std: 0.1089
   CATEs saved to F:\Nexus_Projects\causal-climate-policy\data\processed\cate_estimates.csv


---
## 5.4 — Method Comparison

In [8]:
print("=" * 60)
print("5.4 — Method Comparison")
print("=" * 60)

all_estimates = []
if ols_results:
    all_estimates.extend(ols_results)
if dml_results:
    all_estimates.extend(dml_results)
if cf_ate:
    all_estimates.append(cf_ate)

if all_estimates:
    comparison = compare_methods(all_estimates)
    print(comparison.to_string(index=False))
    
    # Save
    comparison.to_csv(PROJECT_ROOT / "outputs" / "tables" / "method_comparison.csv", index=False)
    print(f"\n Comparison table saved")
    print("  Proceed to Notebook 06: Robustness Checks")

5.4 — Method Comparison
       Method                 Treatment Estimate Std. Error            95% CI p-value    N Significant
          OLS      treat_carbon_pricing   0.1936     0.0196  [0.1551, 0.2321]  0.0000 2028         ***
          OLS   treat_renewable_mandate   0.1055     0.0190  [0.0684, 0.1427]  0.0000 2028         ***
          OLS treat_efficiency_standard   0.2165     0.0251  [0.1672, 0.2657]  0.0000 2028         ***
          OLS                  treat_t1   0.1627     0.0190  [0.1254, 0.1999]  0.0000 2028         ***
          OLS                  treat_t3   0.1181     0.0184  [0.0822, 0.1541]  0.0000 2028         ***
          OLS                  treat_t5   0.0556     0.0178  [0.0208, 0.0904]  0.0017 2028         ***
DML (xgboost)      treat_carbon_pricing   0.0899     0.0154  [0.0598, 0.1200]  0.0000 2028         ***
DML (xgboost)   treat_renewable_mandate   0.0203     0.0143 [-0.0077, 0.0482]  0.1549 2028            
DML (xgboost) treat_efficiency_standard   0.0537 